# 08b - Foundry Agents

This notebook demonstrates **agent creation and invocation** in Azure AI Foundry:

- Semantic mode retrieval (fast, deterministic)
- Agentic mode retrieval (AI-powered query planning)
- YAML-configured agents
- Direct Foundry agent creation with search tools

**Prerequisites:**
- Run `08a-foundry-setup.ipynb` first to configure credentials and connections

**Note:** This notebook assumes the setup cell from 08a has been run.

In [ ]:
# =============================================================================
# Setup - Run this first (loads config from 08a)
# =============================================================================
# If you haven't run 08a-foundry-setup.ipynb, run it first.

import os, sys, json
from pathlib import Path
from dotenv import load_dotenv

# Load .env with override=True to ensure we use .env values (not shell env vars)
load_dotenv(override=True)

print("✅ Environment loaded from .env (override=True)")

# Add notebooks to path for utils
sys.path.insert(0, str(Path.cwd()))

# Import credential utilities including new OBO flow support
from utils import (
    get_user_credential,
    get_agent_credential,
    get_user_token_for_agent,
    get_agent_obo_token,
    AgentOBOCredential,
)

# Load foundry config
with open('foundry-config.json') as f:
    foundry_config = json.load(f)

# Extract key values
# Prefer AI_FOUNDRY_PROJECT_ENDPOINT from .env (required for direct agent creation)
# Fall back to foundry_config endpoint (Azure OpenAI - works for RAG modes only)
PROJECT_ENDPOINT = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT') or foundry_config['foundry']['ai_foundry_endpoint']
MODEL_DEPLOYMENT = foundry_config['foundry'].get('model_deployment', 'gpt-4o')
SEARCH_ENDPOINT = foundry_config['search_endpoint']

# Index names - these match what was created in notebook 05/07
# If your config has them, use them; otherwise use defaults
SEARCH_INDEXES = foundry_config.get('search', {}).get('indexes', {
    'hr': 'hrdocs-index',
    'health': 'healthdocs-index',
})

# Display configuration
print(f"\n📋 Configuration Summary:")
print(f"   AZURE_TENANT_ID: {os.getenv('AZURE_TENANT_ID')[:20]}..." if os.getenv('AZURE_TENANT_ID') else "   ❌ AZURE_TENANT_ID: NOT SET")
print(f"   AZURE_CLIENT_ID: {os.getenv('AZURE_CLIENT_ID')[:20]}..." if os.getenv('AZURE_CLIENT_ID') else "   ❌ AZURE_CLIENT_ID: NOT SET (Agent Blueprint)")
print(f"   AZURE_CLIENT_SECRET: {'✅ Set' if os.getenv('AZURE_CLIENT_SECRET') else '❌ NOT SET'}")

print(f"\n✅ Foundry config loaded from foundry-config.json")
print(f"   Project: {foundry_config['foundry'].get('project_name')}")
print(f"   Model: {MODEL_DEPLOYMENT}")
print(f"   Search: {SEARCH_ENDPOINT}")
print(f"   Indexes: {list(SEARCH_INDEXES.keys())}")

print(f"\n📚 Available Auth Functions:")
print(f"   • get_user_credential()     - Interactive browser auth (for direct access)")
print(f"   • AgentOBOCredential()      - OBO flow credential (agent on behalf of user)")
print(f"   • get_agent_obo_token()     - Get resource token via OBO flow")

## Agent Identity Blueprint OAuth Flows

Agent Identity Blueprints support **three OAuth flows** for different scenarios:

| Flow | Use Case | Documentation |
|------|----------|---------------|
| **On-Behalf-Of (OBO)** | Agent acts on behalf of a signed-in user | [OBO Flow](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/agent-on-behalf-of-oauth-flow) |
| **Autonomous App** | Agent acts autonomously (app-only) | [Autonomous Flow](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/agent-autonomous-app-oauth-flow) |
| **Agent User Impersonation** | Agent impersonates an agent user | [User Flow](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/agent-user-oauth-flow) |

### ⚠️ Key Limitation: No Direct App-Only Tokens

Agent Identity Blueprints **cannot request app-only tokens directly** (error: `AADSTS82001`).

Instead, they must use:
1. **OBO Flow**: Exchange a user token for a resource token
2. **Managed Identity as FIC**: Use a managed identity as a Federated Identity Credential

### Authentication Paths in This Notebook

| Path | Credential | When to Use |
|------|------------|-------------|
| **User Auth** | `InteractiveBrowserCredential` | Direct Azure resource access (Search, Foundry) |
| **Agent OBO** | `AgentOBOCredential` | Agent acting on behalf of signed-in user |

### OBO Flow Steps (Simplified for Notebooks)

```
1. User authenticates → InteractiveBrowserCredential
2. Get token with aud=api://AZURE_CLIENT_ID/access_agent
3. Exchange user token + client secret → Resource token (via OBO)
```

### Required Permissions

**In Azure Portal** → [Entra ID → App Registrations](https://portal.azure.com/#view/Microsoft_AAD_IAM/ActiveDirectoryMenuBlade/~/RegisteredApps):

1. Find your Agent Blueprint (by `AZURE_CLIENT_ID`)
2. **API Permissions** → Add:
   - `Microsoft Graph` → `User.Read` (delegated)
   - `Azure AI Search` scopes (if using search)
3. **Grant admin consent** for your tenant

**RBAC on Azure AI Search** (for your user account):
```bash
USER_ID=$(az ad signed-in-user show --query id -o tsv)
SEARCH_ID=$(az search service show --name <search-name> --resource-group <rg> --query id -o tsv)
az role assignment create --assignee $USER_ID --role "Search Index Data Reader" --scope $SEARCH_ID
```

In [ ]:
# =============================================================================
# Test Agent OBO Flow (Optional)
# =============================================================================
# This cell demonstrates the On-Behalf-Of flow for Agent Identity Blueprints.
# It's optional - the list_search_indexes cell below uses direct user auth.
#
# Uncomment to test OBO flow:

def test_agent_obo_flow():
    """
    Test the Agent OBO flow by:
    1. Getting a user token for the agent scope
    2. Exchanging it for an Azure Search token via OBO
    """
    print("🔐 Testing Agent OBO Flow...")
    print("=" * 60)
    
    try:
        # Step 1: Get user token with agent scope
        print("\n1️⃣  Getting user token for agent scope...")
        user_token = get_user_token_for_agent()
        print(f"   ✅ User token obtained (length: {len(user_token)})")
        
        # Step 2: Exchange for Search token via OBO
        print("\n2️⃣  Exchanging for Azure Search token via OBO...")
        result = get_agent_obo_token(
            resource_scope="https://search.azure.com/.default",
            user_token=user_token,
        )
        print(f"   ✅ OBO token obtained!")
        print(f"   Token type: {result.get('token_type')}")
        print(f"   Expires in: {result.get('expires_in')} seconds")
        
        print("\n" + "=" * 60)
        print("✅ Agent OBO flow working correctly!")
        return True
        
    except Exception as e:
        print(f"\n❌ OBO Flow Error: {e}")
        print("\n💡 Troubleshooting:")
        print("   1. Ensure AZURE_CLIENT_ID and AZURE_CLIENT_SECRET are set in .env")
        print("   2. Run a365.ps1 to configure Agent Blueprint with OAuth scope")
        print("   3. Grant admin consent in Azure Portal:")
        print(f"      https://portal.azure.com/#view/Microsoft_AAD_IAM/ActiveDirectoryMenuBlade/~/RegisteredApps")
        return False

# Uncomment to test:
# test_agent_obo_flow()

In [ ]:
# =============================================================================
# List Available Indexes in Search Service
# =============================================================================
# Run this to verify what indexes exist in your search service
#
# NOTE: We use InteractiveBrowserCredential instead of DefaultAzureCredential
# because DefaultAzureCredential will pick up AZURE_CLIENT_ID from .env,
# which is the Agent Identity Blueprint. Agent identities cannot request
# app-only tokens directly (causes AADSTS82001 error).

import requests
from azure.identity import InteractiveBrowserCredential

# Use InteractiveBrowserCredential to authenticate as YOU (not the agent identity)
TENANT_ID = os.getenv('AZURE_TENANT_ID')
_search_credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)

def list_search_indexes(search_endpoint: str):
    """List all indexes in the search service."""
    token = _search_credential.get_token("https://search.azure.com/.default")
    
    url = f"{search_endpoint}/indexes?api-version=2024-07-01"
    headers = {"Authorization": f"Bearer {token.token}"}
    
    response = requests.get(url, headers=headers)
    if response.ok:
        indexes = response.json().get("value", [])
        print(f"📂 Found {len(indexes)} indexes in {search_endpoint}:")
        for idx in indexes:
            print(f"   • {idx['name']}")
        return [idx['name'] for idx in indexes]
    else:
        print(f"❌ Error listing indexes: {response.status_code}")
        print(f"   {response.text[:200]}")
        return []

available_indexes = list_search_indexes(SEARCH_ENDPOINT)

# Validate configured indexes exist
print(f"\n🔍 Configured indexes:")
for key, name in SEARCH_INDEXES.items():
    exists = "✅" if name in available_indexes else "❌ NOT FOUND"
    print(f"   {key}: {name} {exists}")

## Semantic Mode Retrieval

**Semantic mode** performs hybrid search with semantic ranking:
- Combines keyword + vector search
- Returns top-k documents ranked by relevance
- Fast and deterministic
- Best for: Simple queries with direct answers in documents

In [ ]:
# =============================================================================
# Semantic Mode: Search + Chat Completion
# =============================================================================
# Since we're using Azure OpenAI (not a full AI Foundry project endpoint),
# we use a RAG pattern: search Azure AI Search, then pass results to OpenAI.
#
# This approach works with:
# - Azure OpenAI endpoint (what we have in foundry-config.json)
# - Azure AI Search with RBAC or API key auth

import textwrap
import re
import requests
from openai import AzureOpenAI

# Get Search API key from foundry config (optional - can use RBAC)
SEARCH_API_KEY = foundry_config.get('search_api_key') or os.getenv('AZURE_SEARCH_KEY')
OPENAI_ENDPOINT = foundry_config['openai']['endpoint']
CHAT_DEPLOYMENT = foundry_config['openai']['chat_deployment']

print(f"✅ OpenAI endpoint: {OPENAI_ENDPOINT}")
print(f"   Chat model: {CHAT_DEPLOYMENT}")
print(f"   Search auth: {'API Key' if SEARCH_API_KEY else 'RBAC (user credential)'}")

def extract_source_references(answer_text: str) -> list[str]:
    """Extract source references mentioned in the answer text."""
    sources = []
    seen = set()
    
    # Pattern 1: [Source: xxx] or (Source: xxx)
    pattern1 = r'[\[\(]Source:\s*([^\]\)]+)[\]\)]'
    for match in re.finditer(pattern1, answer_text, re.IGNORECASE):
        src = match.group(1).strip()
        if src not in seen:
            seen.add(src)
            sources.append(src)
    
    # Pattern 2: Markdown links like [title](url) or references like [1], [doc1]
    pattern2 = r'\[([^\]]+)\](?:\([^)]+\))?'
    for match in re.finditer(pattern2, answer_text):
        src = match.group(1).strip()
        # Skip common markdown patterns
        if src.lower() not in ['source', 'note', 'citation', 'ref', 'link'] and src not in seen:
            if len(src) > 2:  # Skip single chars/numbers
                seen.add(src)
                sources.append(src)
    
    return sources

def format_answer_with_sources(answer: str, sources: list = None, width: int = 56):
    """Format the answer with wrapped text and source citations."""
    print("\n" + "=" * 60)
    print("✅ ANSWER")
    print("=" * 60)
    
    wrapped = textwrap.fill(answer, width=width)
    for line in wrapped.split('\n'):
        print(f"   {line}")
    
    if sources:
        print("\n" + "-" * 60)
        print("📑 SOURCES")
        print("-" * 60)
        for i, src in enumerate(sources, 1):
            print(f"   [{i}] {src}")
    
    print("=" * 60 + "\n")

def search_index(
    query: str, 
    index_name: str, 
    top_k: int = 5,
    use_semantic: bool = True,
) -> list[dict]:
    """
    Search Azure AI Search index using simple full-text search.
    
    Returns list of documents with normalized fields:
    - content: the text content (from snippet, content, chunk, or text fields)
    - source: source identifier (from blob_path, source, url, or title fields)
    
    Automatically adapts to different index schemas.
    """
    credential = get_user_credential()
    token = credential.get_token("https://search.azure.com/.default")
    
    headers = {
        "Authorization": f"Bearer {token.token}",
        "Content-Type": "application/json",
    }
    
    # Don't specify $select - retrieve all fields and normalize later
    # This handles indexes with different schemas
    body = {
        "search": query,
        "queryType": "simple",
        "top": top_k,
    }
    
    url = f"{SEARCH_ENDPOINT}/indexes/{index_name}/docs/search?api-version=2024-07-01"
    
    response = requests.post(url, headers=headers, json=body)
    
    if not response.ok:
        print(f"❌ Search error: {response.status_code}")
        error_text = response.text[:300]
        print(f"   {error_text}")
        # Check if index doesn't exist
        if "index" in error_text.lower() and "not found" in error_text.lower():
            print(f"   💡 Index '{index_name}' may not exist. Check available indexes.")
        return []
    
    raw_results = response.json().get("value", [])
    
    # Normalize results to common schema
    # Content field candidates: snippet, content, chunk, text, body, description
    # Source field candidates: blob_path, source, url, title, filepath, filename
    content_fields = ['snippet', 'content', 'chunk', 'text', 'body', 'description']
    source_fields = ['blob_path', 'source', 'url', 'title', 'filepath', 'filename', 'id']
    
    normalized = []
    for doc in raw_results:
        # Find content
        content = None
        for field in content_fields:
            if field in doc and doc[field]:
                content = doc[field]
                break
        
        # Find source
        source = None
        for field in source_fields:
            if field in doc and doc[field]:
                source = doc[field]
                break
        
        # If no content found, concatenate all string fields
        if not content:
            content_parts = []
            for k, v in doc.items():
                if isinstance(v, str) and not k.startswith('@') and not k.endswith('_vector'):
                    content_parts.append(f"{k}: {v}")
            content = "\n".join(content_parts) if content_parts else "(no content)"
        
        if not source:
            source = doc.get('@search.score', 'unknown')
        
        normalized.append({
            'content': content,
            'source': source,
            '_raw': doc,  # Keep original for debugging
        })
    
    return normalized

async def query_semantic_mode(
    query: str,
    index_name: str,
    instructions: str = "You are a helpful assistant. Answer based on the provided context. Always mention which source document you got the information from.",
    show_sources: bool = True,
    top_k: int = 5,
):
    """
    Query using semantic mode: Azure AI Search + Azure OpenAI.
    
    This RAG pattern:
    1. Searches the index with semantic ranking
    2. Builds context from search results
    3. Sends to Azure OpenAI for completion
    """
    print("\n" + "=" * 60)
    print("🔍 SEMANTIC MODE QUERY")
    print("=" * 60)
    print(f"   📂 Index:  {index_name}")
    print(f"   ❓ Query:  {query}")
    print("-" * 60)
    
    # Step 1: Search
    print("   🔎 Searching index...")
    results = search_index(query, index_name, top_k=top_k)
    
    if not results:
        print("   ❌ No search results found")
        return None
    
    print(f"   ✅ Found {len(results)} documents")
    
    # Step 2: Build context from search results
    # Results are normalized with 'content' and 'source' fields
    context_parts = []
    doc_sources = []
    
    for i, doc in enumerate(results, 1):
        # Use normalized fields from search_index
        source_raw = doc.get('source', f'Document {i}')
        source_name = str(source_raw).split('/')[-1] if source_raw else f'Document {i}'
        
        content = doc.get('content', '')[:2000]  # Truncate long content
        
        context_parts.append(f"[Document {i}: {source_name}]\n{content}")
        doc_sources.append(source_name)
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 3: Call Azure OpenAI
    print("   💬 Generating response...")
    
    credential = get_user_credential()
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    
    client = AzureOpenAI(
        azure_endpoint=OPENAI_ENDPOINT,
        azure_ad_token=token.token,
        api_version="2024-06-01",
    )
    
    messages = [
        {"role": "system", "content": f"{instructions}\n\nUse the following documents to answer the user's question:\n\n{context}"},
        {"role": "user", "content": query},
    ]
    
    response = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=messages,
        temperature=0.7,
        max_tokens=1000,
    )
    
    answer = response.choices[0].message.content
    
    # Extract any additional source mentions from the answer
    extracted_sources = extract_source_references(answer) if show_sources else []
    all_sources = list(dict.fromkeys(doc_sources + extracted_sources))  # Dedupe, preserve order
    
    # Format and display
    format_answer_with_sources(answer, all_sources if show_sources else None)
    
    return answer

print("✅ Semantic mode function defined")
print("   Pattern: Azure AI Search (semantic) → Azure OpenAI (chat)")
print(f"   Search: {SEARCH_ENDPOINT}")
print(f"   OpenAI: {OPENAI_ENDPOINT}")

In [ ]:
# Test semantic mode with HR docs
await query_semantic_mode(
    query="What are the company values and mission?",
    index_name=SEARCH_INDEXES['hr'],
    instructions="You are an HR assistant. Answer based on company policies and documentation."
)

In [ ]:
# =============================================================================
# Test Semantic Mode with Health Benefits
# =============================================================================

await query_semantic_mode(
    query="What health insurance plans are available and what do they cover?",
    index_name=SEARCH_INDEXES['health'],
    instructions="You are a benefits specialist. Provide clear, helpful answers about health coverage."
)

## Agentic Mode Retrieval (Foundry IQ)

**Agentic mode** uses AI-powered retrieval with Foundry IQ:
- LLM plans and executes multi-step searches
- Dynamically adjusts queries based on results  
- Higher reasoning for complex questions
- Best for: Complex queries requiring synthesis across documents

**Requirements:**
- Azure OpenAI endpoint URL
- Model deployment for query planning
- Higher latency but better for complex reasoning

### ⚠️ RBAC Requirements for Agentic Mode

Agentic mode requires **Knowledge Sources** in Azure AI Search, which needs specific RBAC roles:

| Role | Purpose |
|------|---------|
| **Search Index Data Contributor** | Read/write index data |
| **Search Service Contributor** | Manage knowledge sources |

To grant these roles via Azure CLI:
```bash
# Get your user object ID
USER_ID=$(az ad signed-in-user show --query id -o tsv)

# Get your search service resource ID
SEARCH_RESOURCE_ID=$(az search service show \
    --name <your-search-service> \
    --resource-group <your-rg> \
    --query id -o tsv)

# Grant Search Index Data Contributor
az role assignment create \
    --assignee $USER_ID \
    --role "Search Index Data Contributor" \
    --scope $SEARCH_RESOURCE_ID

# Grant Search Service Contributor
az role assignment create \
    --assignee $USER_ID \
    --role "Search Service Contributor" \
    --scope $SEARCH_RESOURCE_ID
```

**If you get "Forbidden" errors**, use **Semantic Mode** instead (which works with API keys).

In [ ]:
# =============================================================================
# Agentic Mode: Multi-Step RAG with Query Planning
# =============================================================================
# Since we have Azure OpenAI (not a full AI Foundry project endpoint), we 
# implement "agentic" behavior using multi-step RAG:
# 1. LLM analyzes query and generates sub-queries
# 2. Execute multiple searches
# 3. Synthesize results into final answer
#
# This simulates agentic retrieval without requiring AI Foundry project APIs.

import json

async def query_agentic_mode(
    query: str,
    index_name: str,
    instructions: str = "You are a helpful assistant. Answer based on the provided context. Always mention which source document you got the information from.",
    reasoning_effort: str = "medium",
    show_sources: bool = True,
    max_searches: int = 3,
):
    """
    Query using agentic mode - multi-step RAG with query planning.
    
    This mode:
    - LLM analyzes the query and plans sub-queries
    - Executes multiple searches to gather comprehensive context
    - Synthesizes results into a final answer
    - Best for complex reasoning tasks
    
    Args:
        reasoning_effort: "minimal" (1 search), "low" (2), "medium" (3)
        show_sources: If True, display source documents used
        max_searches: Maximum number of search iterations
    """
    # Map reasoning effort to search count
    search_counts = {"minimal": 1, "low": 2, "medium": 3}
    num_searches = min(search_counts.get(reasoning_effort, 2), max_searches)
    
    print("\n" + "=" * 60)
    print("🤖 AGENTIC MODE QUERY (Multi-Step RAG)")
    print("=" * 60)
    print(f"   📂 Index:     {index_name}")
    print(f"   🧠 Reasoning: {reasoning_effort} ({num_searches} search passes)")
    print(f"   ❓ Query:     {query}")
    print("-" * 60)
    
    credential = get_user_credential()
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    
    client = AzureOpenAI(
        azure_endpoint=OPENAI_ENDPOINT,
        azure_ad_token=token.token,
        api_version="2024-06-01",
    )
    
    all_snippets = []
    all_sources = set()
    search_queries = [query]  # Start with original query
    
    # Step 1: Generate sub-queries if reasoning effort > minimal
    if num_searches > 1:
        print("\n   📝 Planning search strategy...")
        planning_response = client.chat.completions.create(
            model=CHAT_DEPLOYMENT,
            messages=[
                {"role": "system", "content": "You are a search query planner. Given a complex question, break it into 2-3 simpler search queries that together will find all relevant information. Return ONLY a JSON array of query strings, no explanation."},
                {"role": "user", "content": f"Break this question into search queries: {query}"}
            ],
            temperature=0.3,
            max_tokens=200,
        )
        
        try:
            sub_queries = json.loads(planning_response.choices[0].message.content)
            if isinstance(sub_queries, list) and len(sub_queries) > 0:
                search_queries = sub_queries[:num_searches]
                print(f"   ✅ Generated {len(search_queries)} sub-queries:")
                for i, sq in enumerate(search_queries, 1):
                    print(f"      {i}. {sq}")
        except json.JSONDecodeError:
            print("   ⚠️  Could not parse sub-queries, using original query")
    
    # Step 2: Execute searches
    print(f"\n   🔎 Executing {len(search_queries)} search(es)...")
    
    for i, sq in enumerate(search_queries, 1):
        print(f"   [{i}/{len(search_queries)}] Searching: {sq[:50]}...")
        results = search_index(sq, index_name, top_k=3, use_semantic=True)
        
        for doc in results:
            # Use normalized fields from search_index
            source_raw = doc.get('source', '')
            source_name = str(source_raw).split('/')[-1] if source_raw else 'Unknown'
            content = doc.get('content', '')
            
            # Avoid duplicates
            content_key = content[:100]
            if content_key not in [s[:100] for s in all_snippets]:
                all_snippets.append(f"[{source_name}]\n{content}")
                all_sources.add(source_name)
        
        print(f"       Found {len(results)} results")
    
    if not all_snippets:
        print("   ❌ No search results found")
        return None
    
    print(f"\n   📚 Total unique snippets: {len(all_snippets)}")
    print(f"   📄 Sources: {', '.join(all_sources)}")
    
    # Step 3: Synthesize answer
    print("\n   💬 Synthesizing answer...")
    
    context = "\n\n---\n\n".join(all_snippets[:10])  # Limit context size
    
    synthesis_response = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": f"{instructions}\n\nYou have access to the following documents. Synthesize a comprehensive answer that addresses all aspects of the question. Cite specific documents when referencing information.\n\n{context}"},
            {"role": "user", "content": query}
        ],
        temperature=0.7,
        max_tokens=1500,
    )
    
    answer = synthesis_response.choices[0].message.content
    
    # Extract additional source references from answer
    extracted_refs = extract_source_references(answer) if show_sources else []
    final_sources = list(all_sources | set(extracted_refs))
    
    # Format and display
    format_answer_with_sources(answer, final_sources if show_sources else None)
    
    return answer

print("✅ Agentic mode function defined")
print("   Pattern: Query Planning → Multi-Search → Synthesis")
print("   Uses: Azure AI Search + Azure OpenAI (no Foundry project required)")

In [ ]:
# =============================================================================
# Test Agentic Mode with Complex Query (HR Docs)
# =============================================================================

# Test with a complex query that benefits from multi-step reasoning
await query_agentic_mode(
    query="What are the key company policies around employee conduct and how do they relate to the company values?",
    index_name=SEARCH_INDEXES['hr'],
    instructions="You are an HR expert. Synthesize information from multiple policy documents to provide comprehensive answers.",
    reasoning_effort="medium"
)

## Query YAML-Configured Agents

Run queries using agent configurations from the YAML files in `./agents/`.

In [ ]:
# =============================================================================
# Load YAML Agent Configurations
# =============================================================================
# Load agent configs from ./agents/ directory

import yaml
from pathlib import Path

agents_dir = Path("agents")
agent_configs = {}

if agents_dir.exists():
    for yaml_file in agents_dir.glob("*.yaml"):
        with open(yaml_file) as f:
            config = yaml.safe_load(f)
            agent_name = yaml_file.stem  # filename without extension
            agent_configs[agent_name] = config
            print(f"   ✅ Loaded: {agent_name}")

print(f"\n📋 Loaded {len(agent_configs)} agent configurations:")
for name, config in agent_configs.items():
    indexes = config.get('tools', {}).get('azure_ai_search', {}).get('indexes', [])
    index_names = [idx.get('name', 'unknown') for idx in indexes]
    print(f"   • {name}: indexes={index_names}")

In [ ]:
# =============================================================================
# Query Using YAML Agent Configuration
# =============================================================================

async def query_yaml_agent(agent_name: str, query: str, mode: str = "semantic"):
    """
    Query using a YAML-configured agent.
    
    Args:
        agent_name: Name of agent from YAML configs
        query: Question to ask
        mode: "semantic" or "agentic"
    """
    config = agent_configs.get(agent_name)
    if not config:
        print(f"❌ Agent not found: {agent_name}")
        print(f"   Available: {list(agent_configs.keys())}")
        return
    
    # Get index from config
    indexes = config.get('tools', {}).get('azure_ai_search', {}).get('indexes', [])
    if not indexes:
        print(f"❌ No search indexes configured for {agent_name}")
        return
    
    index_name = indexes[0].get('name')
    instructions = config.get('instructions', '')
    
    print(f"\n📋 Agent: {agent_name}")
    print(f"   Mode: {mode}")
    
    if mode == "semantic":
        return await query_semantic_mode(query, index_name, instructions)
    else:
        return await query_agentic_mode(query, index_name, instructions)

print("✅ YAML agent query function defined")

In [ ]:
# =============================================================================
# Test YAML Agents
# =============================================================================
import asyncio

# Test different agents with semantic mode
test_cases = [
    ('fulfillment-agent', 'What is the standard shipping time?'),
    ('regional-apac-agent', 'What are the APAC regional procedures?'),
    ('employee-wellness-agent', 'What health benefits are available?'),
]

print("🧪 Running agent tests...")
print("=" * 60)

for i, (agent_name, query) in enumerate(test_cases):
    if agent_name in agent_configs:
        print(f"\n[{i+1}/{len(test_cases)}] Testing: {agent_name}")
        await query_yaml_agent(agent_name, query, mode="semantic")
        
        # Small delay to allow async cleanup
        await asyncio.sleep(0.5)

print("\n" + "=" * 60)
print("✅ All agent tests completed")
print("=" * 60)

## Summary: Semantic vs Agentic Mode

| Feature | Semantic Mode | Agentic Mode (Foundry IQ) |
|---------|--------------|---------------------------|
| **Search Type** | Hybrid (keyword + vector) | AI-planned multi-step |
| **Ranking** | Semantic reranking | LLM-guided relevance |
| **Speed** | Fast | Higher latency |
| **Best For** | Direct lookups | Complex reasoning |
| **Cost** | Lower | Higher (LLM calls) |

### When to Use Each Mode

**Semantic Mode:**
- FAQ lookups
- Policy questions with direct answers
- High-volume, low-complexity queries

**Agentic Mode (Foundry IQ):**
- Multi-topic synthesis
- Comparative analysis
- Queries requiring reasoning across documents

## Direct Foundry Agent Invocation with Tracing

This section demonstrates how to create and invoke **Foundry Agents** directly using the `azure-ai-projects` SDK with:
- **Azure AI Search Tool** - Connects to our knowledge base indexes
- **OpenTelemetry Tracing** - Shows tool calls, retrieval operations, and response generation

### ⚠️ Prerequisites: AI Foundry Project Endpoint

**These cells require an AI Foundry project endpoint**, not just an Azure OpenAI endpoint.

| Endpoint Type | Example | Works Here? |
|--------------|---------|-------------|
| Azure OpenAI | `https://<name>.openai.azure.com/` | ❌ No |
| AI Foundry Project | `https://<name>.services.ai.azure.com/api/projects/<project>` | ✅ Yes |

**Current endpoint:** Check `foundry_config['foundry']['ai_foundry_endpoint']`

If you only have an Azure OpenAI endpoint, use the **Semantic Mode** and **Agentic Mode** cells above instead.

### Why Direct Agent Invocation?
The Agent Framework's `ChatAgent` provides high-level orchestration, but for deeper visibility into:
- Tool call parameters and results
- Search queries executed against indexes
- Response generation steps

...we use the lower-level `AIProjectClient` with tracing enabled.

In [ ]:
# =============================================================================
# Create Foundry Agent with Azure AI Search Tool
# =============================================================================
# This creates an agent in Azure AI Foundry that can query our knowledge bases
# using the Azure AI Search tool.
#
# IMPORTANT: The search connection in your Foundry project must point to the 
# same search service where your indexes exist. You can add a connection in:
# AI Foundry Portal → Management → Connected Resources → + New Connection

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAISearchAgentTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
    PromptAgentDefinition,
    ConnectionType,
)

# Initialize OpenTelemetry tracer for agent operations
from opentelemetry import trace
tracer = trace.get_tracer(__name__)

# Check if we have an AI Foundry project endpoint (not just Azure OpenAI)
_is_foundry_project = "services.ai.azure.com" in PROJECT_ENDPOINT or "/api/projects/" in PROJECT_ENDPOINT
print(f"📋 Endpoint: {PROJECT_ENDPOINT}")
if _is_foundry_project:
    print("   ✅ AI Foundry project endpoint detected")
else:
    print("   ⚠️  This appears to be an Azure OpenAI endpoint, not an AI Foundry project")
    print("   ⚠️  Direct agent creation may fail - use Semantic/Agentic mode cells above instead")

async def create_foundry_agent_with_search(
    agent_name: str,
    instructions: str,
    index_name: str,
    search_connection_id: str = None,
    expected_search_name: str = None,  # Optional: verify we're using the right search service
):
    """
    Create a Foundry agent with Azure AI Search tool.
    
    Args:
        agent_name: Name for the agent
        instructions: System instructions for the agent
        index_name: Azure AI Search index to connect
        search_connection_id: Project connection ID for search (optional, auto-discovered)
        expected_search_name: Expected search service name to verify connection (optional)
    
    Returns:
        Tuple of (agent, project_client, openai_client)
    """
    credential = get_user_credential()
    
    with tracer.start_as_current_span(f"create_agent_{agent_name}") as span:
        span.set_attribute("agent.name", agent_name)
        span.set_attribute("search.index", index_name)
        
        # Create the project client
        project_client = AIProjectClient(
            endpoint=PROJECT_ENDPOINT,
            credential=credential,
        )
        
        # Get connections to find search connection ID
        if not search_connection_id:
            print("📡 Finding Azure AI Search connection...")
            connections = list(project_client.connections.list())
            
            search_connections = []
            for conn in connections:
                conn_type = getattr(conn, 'type', None) or getattr(conn, 'connection_type', 'unknown')
                print(f"   Found connection: {conn.name} ({conn_type})")
                
                # Check for AI Search connection by name or type
                is_search = (
                    "search" in conn.name.lower() or 
                    conn_type == ConnectionType.AZURE_AI_SEARCH or
                    str(conn_type).lower() == "azureaisearch"
                )
                if is_search:
                    search_connections.append(conn)
            
            # If expected_search_name is provided, try to match it
            if expected_search_name and search_connections:
                for conn in search_connections:
                    if expected_search_name.lower() in conn.name.lower():
                        search_connection_id = conn.id
                        print(f"   ✅ Using (matched): {conn.name}")
                        break
            
            # Fall back to first search connection
            if not search_connection_id and search_connections:
                search_connection_id = search_connections[0].id
                print(f"   ✅ Using: {search_connections[0].name}")
                
                # Warn if there are multiple search connections
                if len(search_connections) > 1:
                    print(f"   ⚠️  Found {len(search_connections)} search connections - using first one")
                    print(f"      If your index is on a different search service, add it as a connection in AI Foundry")
        
        if not search_connection_id:
            print("⚠️  No Azure AI Search connection found in project")
            print("   Add one in AI Foundry portal → Management → Connected Resources")
            print(f"   Your search service: {SEARCH_ENDPOINT}")
            return None, None, None
        
        # Create Azure AI Search tool
        search_tool = AzureAISearchAgentTool(
            azure_ai_search=AzureAISearchToolResource(
                indexes=[
                    AISearchIndexResource(
                        project_connection_id=search_connection_id,
                        index_name=index_name,
                        query_type=AzureAISearchQueryType.SEMANTIC,
                    ),
                ]
            )
        )
        
        # Get OpenAI client for agent operations
        openai_client = project_client.get_openai_client()
        
        # Create the agent with search tool
        print(f"\n🤖 Creating agent: {agent_name}")
        print(f"   Using index: {index_name}")
        agent = project_client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(
                model=MODEL_DEPLOYMENT,
                instructions=instructions,
                tools=[search_tool],
            ),
        )
        print(f"   ✅ Agent created (id: {agent.id}, version: {agent.version})")
        
        return agent, project_client, openai_client

print("✅ Agent creation function defined")
print("   Uses: AIProjectClient, AzureAISearchAgentTool")
print(f"\n📋 Your search service: {SEARCH_ENDPOINT}")
print(f"   Make sure this search service is connected in AI Foundry portal")

In [ ]:
# =============================================================================
# Invoke Foundry Agent with Tracing
# =============================================================================
# This function invokes the agent and captures all tool calls in traces

async def invoke_foundry_agent(
    agent,
    openai_client,
    project_client,
    query: str,
    show_tool_calls: bool = True,
):
    """
    Invoke a Foundry agent and show tool calls.
    
    Args:
        agent: The agent to invoke
        openai_client: OpenAI client from project
        project_client: AIProjectClient for cleanup
        query: User query
        show_tool_calls: Whether to print tool call details
    
    Returns:
        Response text
    """
    with tracer.start_as_current_span("invoke_agent") as span:
        span.set_attribute("query", query)
        span.set_attribute("agent.name", agent.name)
        
        print("\n" + "=" * 60)
        print("🔍 FOUNDRY AGENT INVOCATION")
        print("=" * 60)
        print(f"   Agent: {agent.name}")
        print(f"   Query: {query}")
        print("-" * 60)
        
        # Create a conversation
        conversation = openai_client.conversations.create(
            items=[{"type": "message", "role": "user", "content": query}],
        )
        print(f"   📝 Conversation: {conversation.id}")
        
        # Invoke agent - NOTE: Don't specify model when using agent reference
        response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
            input="",  # Query is in conversation
        )
        
        # Extract tool calls from response output
        if show_tool_calls:
            print("\n📊 TOOL CALLS:")
            print("-" * 40)
            tool_calls_found = False
            
            for output in response.output:
                output_type = getattr(output, 'type', None)
                
                if output_type == 'function_call':
                    tool_calls_found = True
                    print(f"   🔧 Function: {output.name}")
                    print(f"      Arguments: {output.arguments}")
                    
                elif output_type == 'azure_search_call':
                    tool_calls_found = True
                    print(f"   🔍 Azure Search Call:")
                    if hasattr(output, 'queries'):
                        for q in output.queries:
                            print(f"      Query: {q}")
                    if hasattr(output, 'results'):
                        print(f"      Results: {len(output.results)} documents")
                        
                elif output_type == 'message':
                    # Skip message outputs in tool call section
                    pass
                    
                else:
                    if output_type and output_type not in ['message']:
                        tool_calls_found = True
                        print(f"   📌 {output_type}: {output}")
            
            if not tool_calls_found:
                print("   (No explicit tool calls in response)")
        
        # Display response
        print("\n💬 RESPONSE:")
        print("-" * 40)
        print(response.output_text)
        
        # Cleanup conversation
        openai_client.conversations.delete(conversation_id=conversation.id)
        
        return response.output_text

print("✅ Agent invocation function defined")
print("   Captures: conversations, tool calls, responses")

In [ ]:
# =============================================================================
# Test: Create and Invoke HR Agent with Search
# =============================================================================
# Creates an agent connected to the hrdocs-index and queries it
# Note: We specifically use 'a365-search-connection' which should point to our search service

# Skip if we don't have an AI Foundry project endpoint
if not _is_foundry_project:
    print("⚠️  Skipping: This cell requires an AI Foundry project endpoint")
    print("   Current endpoint is Azure OpenAI, not AI Foundry project")
    print("   Use the Semantic Mode or Agentic Mode cells above instead")
    hr_agent, project_client, openai_client = None, None, None
else:
    # Create the HR agent - use expected_search_name to match our search service connection
    hr_agent, project_client, openai_client = await create_foundry_agent_with_search(
        agent_name="hr-knowledge-agent",
        instructions="""You are an HR specialist for Zava company. 
Answer questions about company policies, benefits, and procedures.
Always cite the source documents when providing information.
Be concise but thorough.""",
        index_name=SEARCH_INDEXES['hr'],  # hrdocs-index
        expected_search_name="a365-search",  # Match our search service connection name
    )

    if hr_agent:
        # Test query
        await invoke_foundry_agent(
            hr_agent,
            openai_client,
            project_client,
            query="What are the company's policies on remote work and flexible hours?",
            show_tool_calls=True,
        )
        
        # Cleanup (uncomment to delete agent after test)
        # project_client.agents.delete_version(agent_name=hr_agent.name, agent_version=hr_agent.version)
        # print("\\n🗑️  Agent cleaned up")

In [ ]:
# =============================================================================
# Test: Create and Invoke Health Benefits Agent
# =============================================================================
# Creates an agent connected to the healthdocs-index and queries it

# Skip if we don't have an AI Foundry project endpoint
if not _is_foundry_project:
    print("⚠️  Skipping: This cell requires an AI Foundry project endpoint")
    print("   Use the Semantic Mode or Agentic Mode cells above instead")
    health_agent, project_client, openai_client = None, None, None
else:
    health_agent, project_client, openai_client = await create_foundry_agent_with_search(
        agent_name="health-benefits-agent",
        instructions="""You are a health benefits specialist.
Answer questions about health insurance plans, coverage, and benefits.
Provide specific details from the policy documents.
If information is not available, say so clearly.""",
        index_name=SEARCH_INDEXES['health'],  # healthdocs-index
        expected_search_name="a365-search",  # Match our search service connection name
    )

    if health_agent:
        # Test query
        await invoke_foundry_agent(
            health_agent,
            openai_client,
            project_client,
            query="What mental health services are covered under the health insurance plan?",
            show_tool_calls=True,
        )
        
        # Cleanup (uncomment to delete agent after test)
        # project_client.agents.delete_version(agent_name=health_agent.name, agent_version=health_agent.version)
        # print("\\n🗑️  Agent cleaned up")

### Understanding the Traces

When you run the agent invocations above, OpenTelemetry captures:

| Span | Description |
|------|-------------|
| `create_agent_*` | Agent creation with search tool configuration |
| `invoke_agent` | Full agent invocation including tool calls |
| Azure AI Search calls | Queries sent to the search index |
| Response generation | LLM processing and response creation |

**Tool Calls in Response:**
- `azure_search_call` - Search queries against the index
- `function_call` - Custom function invocations
- `message` - Final response text

For production monitoring, connect to **Azure Monitor Application Insights**:
```python
from azure.monitor.opentelemetry import configure_azure_monitor
app_insights_conn = project_client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=app_insights_conn)
```